# Amharic Speech-to-Speech Assistant

## Final reproducible pipeline

This notebook contains the final inference architecture selected after the ASR, LLM fine-tuning, safety, TTS, and latency experiments.

**Pipeline**

`Microphone / WAV → Ethio-ASR → transcript-quality guard → critical-information guard → Gemma 3 4B V4 LoRA → boundary-safe response chunking → MMS-TTS → Amharic audio`

The final language-model checkpoint is **Gemma 3 4B + V4 LoRA, checkpoint 118**. The notebook is inference-only: it does not rebuild the training datasets or retrain V1–V4.

## Experiment summary

| Stage | Result |
|---|---|
| ASR selection | Generic and adapted Whisper variants were explored; `badrex/Ethio-ASR-multilingual-94M` was selected for its speed and Amharic coverage. |
| V1 | 982 training examples. Removed most foreign-script leakage and improved critical date/time behavior, but over-clarified noisy inputs. |
| V2 | 993 examples. Reduced over-clarification but introduced semantic flattening in some technical inputs. |
| V3 | 969 examples. Improved language purity; technical semantic recovery remained inconsistent. |
| V4 | 1,046 examples. Added targeted technical recovery and critical-number contrastive examples. This is the final fine-tuned model. |
| Locked model test | T01–T24: approximately 15 clear passes, 2 partials, and 7 failures. Stronger technical generalization, but critical names/dates/numbers remained a safety weakness. |
| Safety layer | Added transcript-quality and critical-information guards rather than relying on fine-tuning alone. |
| TTS optimization | Persistent Uroman preprocessing removed the repeated tokenizer initialization overhead. |
| Streaming experiments | Concurrent LLM+TTS caused GPU contention. Four-word chunking reduced first-audio latency but introduced a long inter-chunk gap. Boundary-safe six-word time-slicing was retained. |

### Selected benchmark snapshots

| Measurement | Observed result |
|---|---:|
| Warm Ethio-ASR total latency on P07 | approximately 0.04–0.05 s |
| P07 warm full-pipeline median before time-slicing | 5.909 s |
| Four-word early-audio experiment | 3.063 s first audio, 2.782 s predicted gap |
| Six-word development experiment on P03 | 2.570 s median first-audio-ready, 0.000 s predicted gap |
| P09 final evaluation ASR | 0.067 s |
| P09 final evaluation first-audio-ready | 5.694 s |

These are experiment snapshots, not a final aggregate benchmark. `first_audio_ready_seconds` measures compute time until the first response waveform is ready; it is not the browser or speaker playback timestamp.

## 1. Environment

Use a Colab GPU runtime. The first cell installs the runtime dependencies. If Colab asks for a runtime restart after installation, restart once and continue from the imports cell.

In [1]:
%pip install -q unsloth soundfile librosa uroman scipy sentencepiece gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.4/25.4 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.7/930.7 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/

In [2]:
import unsloth

import json
import os
import re
import time
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
import torch
import uroman

from transformers import (
    AutoModelForCTC,
    AutoProcessor,
    AutoTokenizer,
    StoppingCriteria,
    StoppingCriteriaList,
    VitsModel,
)

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU runtime is required.")

DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4


## 2. Model location

Mount Google Drive and point `V4_ADAPTER_DIR` to the shared copy of `gemma3_amh_asr_lora_v4`.

Only this path should need to change when the project is moved to the team folder.

In [3]:
from google.colab import drive

drive.mount("/content/drive")

V4_ADAPTER_DIR = Path(
    "/content/drive/MyDrive/amharic_s2s_finetuning/"
    "gemma3_4b_asr_robustness_v1/gemma3_amh_asr_lora_v4"
)

if not (V4_ADAPTER_DIR / "adapter_config.json").exists():
    raise FileNotFoundError(
        "Final V4 adapter not found. Update V4_ADAPTER_DIR to the shared-drive model folder."
    )

print("Adapter:", V4_ADAPTER_DIR)

Mounted at /content/drive


FileNotFoundError: Final V4 adapter not found. Update V4_ADAPTER_DIR to the shared-drive model folder.

## 3. Load the final models

In [ ]:
from unsloth import FastModel

MODEL_NAME = "unsloth/gemma-3-4b-it-unsloth-bnb-4bit"
ASR_MODEL_NAME = "badrex/Ethio-ASR-multilingual-94M"
TTS_MODEL_NAME = "facebook/mms-tts-amh"

model_v4, tokenizer_v4 = FastModel.from_pretrained(
    model_name=str(V4_ADAPTER_DIR),
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)
FastModel.for_inference(model_v4)

asr_processor = AutoProcessor.from_pretrained(ASR_MODEL_NAME)
asr_model = AutoModelForCTC.from_pretrained(ASR_MODEL_NAME).to(DEVICE).eval()
ASR_SR = asr_processor.feature_extractor.sampling_rate

tts_tokenizer = AutoTokenizer.from_pretrained(TTS_MODEL_NAME)
tts_model = VitsModel.from_pretrained(TTS_MODEL_NAME).to(DEVICE).eval()
TTS_SR = tts_model.config.sampling_rate

uromanizer = uroman.Uroman()

print("Gemma V4, Ethio-ASR, and MMS-TTS loaded.")

## 4. Assistant prompt and safety guards

In [ ]:
SYSTEM_PROMPT = """
You are a short-response Amharic voice assistant.

The user's message comes from automatic speech recognition and may
contain transcription errors.

Rules:
- Respond naturally and briefly in Amharic.
- Do not simply repeat the user's message.
- Never invent missing information.
- Minor transcription errors may be interpreted if the intended meaning is clear.
- If a critical name, date, time, number, amount, or place is unclear,
  ask the user to repeat only that information.
- If the whole message is unintelligible, ask the user to repeat it.
- Understand common Amharic-English code-switching.
- Do not explain your reasoning.
""".strip()

DATE_TIME_TERMS = [
    "ቀን", "ሰዓት", "ጠዋት", "ከጠዋቱ", "ከሰዓት", "ማታ",
    "መስከረም", "ጥቅምት", "ህዳር", "ታህሳስ", "ጥር", "የካቲት",
    "መጋቢት", "ሚያዝያ", "ግንቦት", "ሰኔ", "ሐምሌ", "ነሐሴ",
    "ጳጉሜ", "date", "time",
]
AMOUNT_TERMS = ["ብር", "ዋጋ", "ዋጋው", "መጠን", "amount", "price"]
QUANTITY_TERMS = ["ሳጥን", "ኬዝ", "case", "cases", "quantity", "qty", "%"]
PERSON_CONTEXT_TERMS = ["ጋር", "አላት", "አላቸው", "ልኮታል", "ላከ", "ላከች", "ስም"]

REPEAT_RESPONSE = "ይቅርታ፣ ንግግሩ ግልጽ አልነበረም። እባክዎን እንደገና ይናገሩ።"


def contains_any(text, terms):
    lower = text.lower()
    return any(term.lower() in lower for term in terms)


def suspicious_tokens(text):
    text = text.strip()
    scan_text = re.sub(r"\?\s*$", "", text)
    tokens = re.findall(r"\S+", scan_text)
    return [token for token in tokens if re.search(r"[?xX�]", token)]


def critical_information_guard(text):
    text = text.strip()
    bad_tokens = suspicious_tokens(text)

    if not bad_tokens:
        return {"blocked": False, "reason": None, "response": None, "tokens": []}

    if contains_any(text, AMOUNT_TERMS):
        return {
            "blocked": True,
            "reason": "uncertain_amount",
            "response": "መጠኑ ግልጽ አልነበረም። እባክዎን መጠኑን ብቻ እንደገና ይናገሩ።",
            "tokens": bad_tokens,
        }

    if contains_any(text, QUANTITY_TERMS):
        return {
            "blocked": True,
            "reason": "uncertain_quantity",
            "response": "ቁጥሩ ግልጽ አልነበረም። እባክዎን ቁጥሩን ብቻ እንደገና ይናገሩ።",
            "tokens": bad_tokens,
        }

    if contains_any(text, DATE_TIME_TERMS):
        return {
            "blocked": True,
            "reason": "uncertain_date_time",
            "response": "የቀን ወይም የሰዓት መረጃው ግልጽ አልነበረም። እባክዎን ቀኑንና ሰዓቱን እንደገና ይናገሩ።",
            "tokens": bad_tokens,
        }

    if contains_any(text, PERSON_CONTEXT_TERMS):
        return {
            "blocked": True,
            "reason": "uncertain_person_or_name",
            "response": "ስሙ ግልጽ አልነበረም። እባክዎን ስሙን ብቻ እንደገና ይናገሩ።",
            "tokens": bad_tokens,
        }

    return {
        "blocked": True,
        "reason": "explicit_uncertainty",
        "response": "የተናገሩት አንድ ክፍል ግልጽ አልነበረም። እባክዎን ያንን ክፍል እንደገና ይናገሩ።",
        "tokens": bad_tokens,
    }


def is_ethiopic_char(ch):
    cp = ord(ch)
    return (
        0x1200 <= cp <= 0x137F
        or 0x1380 <= cp <= 0x139F
        or 0x2D80 <= cp <= 0x2DDF
        or 0xAB00 <= cp <= 0xAB2F
    )


def transcript_quality_guard(text, lid=None):
    text = text.strip()

    if not text:
        return {"blocked": True, "reason": "empty_transcript", "response": REPEAT_RESPONSE}

    alpha_chars = [ch for ch in text if ch.isalpha()]
    ethiopic_chars = [ch for ch in alpha_chars if is_ethiopic_char(ch)]
    tokens = text.split()

    ascii_alpha_only = (
        len(alpha_chars) > 0
        and len(ethiopic_chars) == 0
        and all(ord(ch) < 128 for ch in alpha_chars)
    )

    short_non_amharic_noise = (
        ascii_alpha_only
        and len(tokens) <= 2
        and len(alpha_chars) <= 20
        and lid is not None
        and lid != "AMH"
    )

    if short_non_amharic_noise:
        return {
            "blocked": True,
            "reason": "short_non_amharic_noise",
            "response": REPEAT_RESPONSE,
        }

    return {"blocked": False, "reason": None, "response": None}

## 5. ASR and TTS utilities

In [ ]:
LID_PATTERN = re.compile(r"^\[(AMH|TIR|ORM|SID|WAL)\]\s*")


def fast_tts_tokenize(text):
    normalized = tts_tokenizer.normalize_text(text)
    normalized = tts_tokenizer._preprocess_char(normalized)
    romanized = uromanizer.romanize_string(normalized)
    inputs = tts_tokenizer(romanized, return_tensors="pt", normalize=True)
    return inputs, romanized


def transcribe_ethio_asr(audio_path):
    total_start = time.perf_counter()
    prep_start = time.perf_counter()

    audio, sr = sf.read(audio_path, dtype="float32")
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    if sr != ASR_SR:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=ASR_SR)
        sr = ASR_SR

    inputs = asr_processor(audio, sampling_rate=sr, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    prep_seconds = time.perf_counter() - prep_start

    torch.cuda.synchronize()
    inference_start = time.perf_counter()
    with torch.inference_mode():
        logits = asr_model(**inputs).logits
    pred_ids = torch.argmax(logits, dim=-1)
    raw_transcript = asr_processor.batch_decode(pred_ids)[0].strip()
    torch.cuda.synchronize()
    inference_seconds = time.perf_counter() - inference_start

    lid_match = LID_PATTERN.match(raw_transcript)
    lid = lid_match.group(1) if lid_match else None
    transcript = LID_PATTERN.sub("", raw_transcript).strip()

    return {
        "raw_transcript": raw_transcript,
        "transcript": transcript,
        "lid": lid,
        "audio_seconds": len(audio) / sr,
        "prep_seconds": prep_seconds,
        "inference_seconds": inference_seconds,
        "total_seconds": time.perf_counter() - total_start,
    }


def synthesize_mms(text, seed=3407):
    total_start = time.perf_counter()
    tokenize_start = time.perf_counter()
    inputs, romanized = fast_tts_tokenize(text)
    tokenize_seconds = time.perf_counter() - tokenize_start
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.cuda.synchronize()
    synth_start = time.perf_counter()
    with torch.inference_mode():
        waveform = tts_model(**inputs).waveform[0]
    torch.cuda.synchronize()
    synth_seconds = time.perf_counter() - synth_start

    waveform = waveform.detach().float().cpu().numpy()

    return {
        "waveform": waveform,
        "sampling_rate": TTS_SR,
        "romanized": romanized,
        "tokenize_seconds": tokenize_seconds,
        "synth_seconds": synth_seconds,
        "total_seconds": time.perf_counter() - total_start,
        "audio_seconds": len(waveform) / TTS_SR,
    }

## 6. Boundary-safe time-sliced Gemma generation

In [ ]:
WORD_END_CHARS = set(" \n\t፣።፤፥፦.,!?;:")


def stable_text_prefix(text):
    last_boundary = -1
    for i, ch in enumerate(text):
        if ch in WORD_END_CHARS:
            last_boundary = i
    if last_boundary < 0:
        return ""
    return text[: last_boundary + 1].strip()


class StopAfterStableWords(StoppingCriteria):
    def __init__(self, tokenizer, prompt_length, min_stable_words=6):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length
        self.min_stable_words = min_stable_words

    def __call__(self, input_ids, scores, **kwargs):
        generated = input_ids[0, self.prompt_length :]
        text = self.tokenizer.decode(generated, skip_special_tokens=True)
        stable = stable_text_prefix(text)
        return bool(stable) and len(stable.split()) >= self.min_stable_words


def build_chat_inputs(text):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "text", "text": text}]},
    ]
    inputs = tokenizer_v4.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    return {k: v.to(model_v4.device) for k, v in inputs.items()}


def generate_first_stage(text, min_words=6):
    inputs = build_chat_inputs(text)
    prompt_length = inputs["input_ids"].shape[-1]

    stopping = StoppingCriteriaList(
        [StopAfterStableWords(tokenizer_v4, prompt_length, min_stable_words=min_words)]
    )

    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        output = model_v4.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            repetition_penalty=1.05,
            use_cache=True,
            stopping_criteria=stopping,
        )
    torch.cuda.synchronize()

    elapsed = time.perf_counter() - start
    generated_ids = output[0, prompt_length:].detach()
    generated_text = tokenizer_v4.decode(generated_ids, skip_special_tokens=True).strip()
    spoken_chunk = stable_text_prefix(generated_text) or generated_text
    attention_mask = inputs.get("attention_mask", torch.ones_like(inputs["input_ids"]))

    return {
        "spoken_chunk": spoken_chunk,
        "generated_so_far": generated_text,
        "generation_seconds": elapsed,
        "prefix_ids": generated_ids,
        "prompt_input_ids": inputs["input_ids"].detach(),
        "prompt_attention_mask": attention_mask.detach(),
    }


def continue_v4_from_prefix(state, max_new_tokens=24):
    prompt_ids = state["prompt_input_ids"].to(model_v4.device)
    prompt_mask = state["prompt_attention_mask"].to(model_v4.device)
    prefix_ids = state["prefix_ids"].unsqueeze(0).to(model_v4.device)
    prefix_mask = torch.ones(
        prefix_ids.shape,
        dtype=prompt_mask.dtype,
        device=model_v4.device,
    )

    combined_ids = torch.cat([prompt_ids, prefix_ids], dim=-1)
    combined_mask = torch.cat([prompt_mask, prefix_mask], dim=-1)
    input_length = combined_ids.shape[-1]

    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        output = model_v4.generate(
            input_ids=combined_ids,
            attention_mask=combined_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            use_cache=True,
        )
    torch.cuda.synchronize()

    elapsed = time.perf_counter() - start
    new_ids = output[0, input_length:].detach()
    all_generated_ids = torch.cat([state["prefix_ids"].to(new_ids.device), new_ids], dim=0)
    full_generated_text = tokenizer_v4.decode(
        all_generated_ids, skip_special_tokens=True
    ).strip()

    spoken_chunk = state["spoken_chunk"]
    if full_generated_text.startswith(spoken_chunk):
        remainder_text = full_generated_text[len(spoken_chunk):].lstrip()
    else:
        remainder_text = full_generated_text

    return {
        "remainder_text": remainder_text,
        "full_response": full_generated_text,
        "seconds": elapsed,
        "generated_tokens": new_ids.shape[-1],
    }

## 7. Warm-up and cached guard audio

In [ ]:
warm_44100 = np.zeros(44100, dtype=np.float32)
_ = librosa.resample(warm_44100, orig_sr=44100, target_sr=ASR_SR)

warm_audio = np.zeros(ASR_SR, dtype=np.float32)
warm_inputs = asr_processor(
    warm_audio,
    sampling_rate=ASR_SR,
    return_tensors="pt",
)
warm_inputs = {k: v.to(DEVICE) for k, v in warm_inputs.items()}
with torch.inference_mode():
    _ = asr_model(**warm_inputs).logits

_ = generate_first_stage(
    "ከአዲስ አበባ ወደ ባህር ዳር በመኪና መሄድ ጊዜ ይወስዳል።",
    min_words=6,
)
_ = synthesize_mms("እሺ፣ ገብቶኛል።")

GUARD_RESPONSES = [
    REPEAT_RESPONSE,
    "መጠኑ ግልጽ አልነበረም። እባክዎን መጠኑን ብቻ እንደገና ይናገሩ።",
    "ቁጥሩ ግልጽ አልነበረም። እባክዎን ቁጥሩን ብቻ እንደገና ይናገሩ።",
    "የቀን ወይም የሰዓት መረጃው ግልጽ አልነበረም። እባክዎን ቀኑንና ሰዓቱን እንደገና ይናገሩ።",
    "ስሙ ግልጽ አልነበረም። እባክዎን ስሙን ብቻ እንደገና ይናገሩ።",
    "የተናገሩት አንድ ክፍል ግልጽ አልነበረም። እባክዎን ያንን ክፍል እንደገና ይናገሩ።",
]

GUARD_AUDIO_CACHE = {}
for response in GUARD_RESPONSES:
    tts = synthesize_mms(response)
    GUARD_AUDIO_CACHE[response] = {
        "waveform": tts["waveform"],
        "sampling_rate": tts["sampling_rate"],
        "audio_seconds": tts["audio_seconds"],
    }

torch.cuda.empty_cache()
print("Warm-up complete. Guard responses cached:", len(GUARD_AUDIO_CACHE))

## 8. Final frozen speech-to-speech pipeline

In [ ]:
def run_final_s2s(audio_path, reference_text=None):
    overall_start = time.perf_counter()
    asr = transcribe_ethio_asr(audio_path)
    transcript = asr["transcript"]
    lid = asr["lid"]

    quality_start = time.perf_counter()
    quality = transcript_quality_guard(transcript, lid)
    quality_guard_seconds = time.perf_counter() - quality_start

    if quality["blocked"]:
        response = quality["response"]
        cached = GUARD_AUDIO_CACHE[response]
        return {
            "reference": reference_text,
            "raw_asr": asr["raw_transcript"],
            "transcript": transcript,
            "lid": lid,
            "response_source": "quality_guard",
            "guard_reason": quality["reason"],
            "response": response,
            "first_chunk": response,
            "remainder": "",
            "asr_seconds": asr["total_seconds"],
            "quality_guard_seconds": quality_guard_seconds,
            "critical_guard_seconds": 0.0,
            "first_chunk_llm_seconds": 0.0,
            "first_tts_seconds": 0.0,
            "first_audio_ready_seconds": time.perf_counter() - overall_start,
            "first_audio_duration": cached["audio_seconds"],
            "continuation_llm_seconds": 0.0,
            "remainder_tts_seconds": 0.0,
            "predicted_gap_seconds": 0.0,
            "first_audio": cached["waveform"],
            "first_audio_sr": cached["sampling_rate"],
            "remainder_audio": None,
            "remainder_audio_sr": None,
        }

    critical_start = time.perf_counter()
    critical = critical_information_guard(transcript)
    critical_guard_seconds = time.perf_counter() - critical_start

    if critical["blocked"]:
        response = critical["response"]
        cached = GUARD_AUDIO_CACHE[response]
        return {
            "reference": reference_text,
            "raw_asr": asr["raw_transcript"],
            "transcript": transcript,
            "lid": lid,
            "response_source": "critical_guard",
            "guard_reason": critical["reason"],
            "response": response,
            "first_chunk": response,
            "remainder": "",
            "asr_seconds": asr["total_seconds"],
            "quality_guard_seconds": quality_guard_seconds,
            "critical_guard_seconds": critical_guard_seconds,
            "first_chunk_llm_seconds": 0.0,
            "first_tts_seconds": 0.0,
            "first_audio_ready_seconds": time.perf_counter() - overall_start,
            "first_audio_duration": cached["audio_seconds"],
            "continuation_llm_seconds": 0.0,
            "remainder_tts_seconds": 0.0,
            "predicted_gap_seconds": 0.0,
            "first_audio": cached["waveform"],
            "first_audio_sr": cached["sampling_rate"],
            "remainder_audio": None,
            "remainder_audio_sr": None,
        }

    stage1 = generate_first_stage(transcript, min_words=6)
    first_tts = synthesize_mms(stage1["spoken_chunk"])
    first_audio_ready = time.perf_counter() - overall_start

    stage2 = continue_v4_from_prefix(stage1, max_new_tokens=24)
    remainder_text = stage2["remainder_text"].strip()

    if remainder_text:
        remainder_tts = synthesize_mms(remainder_text)
        remainder_audio = remainder_tts["waveform"]
        remainder_sr = remainder_tts["sampling_rate"]
        remainder_tts_seconds = remainder_tts["total_seconds"]
    else:
        remainder_audio = None
        remainder_sr = None
        remainder_tts_seconds = 0.0

    continuation_compute = stage2["seconds"] + remainder_tts_seconds
    predicted_gap = max(0.0, continuation_compute - first_tts["audio_seconds"])

    return {
        "reference": reference_text,
        "raw_asr": asr["raw_transcript"],
        "transcript": transcript,
        "lid": lid,
        "response_source": "v4_streaming",
        "guard_reason": None,
        "response": stage2["full_response"],
        "first_chunk": stage1["spoken_chunk"],
        "remainder": remainder_text,
        "asr_seconds": asr["total_seconds"],
        "quality_guard_seconds": quality_guard_seconds,
        "critical_guard_seconds": critical_guard_seconds,
        "first_chunk_llm_seconds": stage1["generation_seconds"],
        "first_tts_seconds": first_tts["total_seconds"],
        "first_audio_ready_seconds": first_audio_ready,
        "first_audio_duration": first_tts["audio_seconds"],
        "continuation_llm_seconds": stage2["seconds"],
        "remainder_tts_seconds": remainder_tts_seconds,
        "predicted_gap_seconds": predicted_gap,
        "first_audio": first_tts["waveform"],
        "first_audio_sr": first_tts["sampling_rate"],
        "remainder_audio": remainder_audio,
        "remainder_audio_sr": remainder_sr,
    }


def combine_response_audio(result):
    first = result["first_audio"]
    remainder = result["remainder_audio"]
    sr = result["first_audio_sr"]

    if remainder is None:
        return sr, first

    if result["remainder_audio_sr"] != sr:
        remainder = librosa.resample(
            remainder,
            orig_sr=result["remainder_audio_sr"],
            target_sr=sr,
        )

    gap_samples = int(round(result["predicted_gap_seconds"] * sr))
    gap = np.zeros(gap_samples, dtype=np.float32)
    combined = np.concatenate([first, gap, remainder]).astype(np.float32)
    return sr, combined


def result_metrics_text(result):
    return (
        f"Route: {result['response_source']}\\n"
        f"ASR: {result['asr_seconds']:.3f} s\\n"
        f"First audio ready: {result['first_audio_ready_seconds']:.3f} s\\n"
        f"Predicted inter-chunk gap: {result['predicted_gap_seconds']:.3f} s"
    )

## 9. File-based evaluation

Use this helper for benchmark recordings. Keep the reference sentence written down before running the sample. Evaluation samples should be logged rather than used for further model or architecture tuning.

In [ ]:
from IPython.display import Audio, display


def evaluate_file(audio_path, reference_text=None):
    result = run_final_s2s(audio_path, reference_text=reference_text)
    sr, combined_audio = combine_response_audio(result)

    print("Reference:", result["reference"])
    print("ASR:", result["transcript"])
    print("LID:", result["lid"])
    print("Route:", result["response_source"])
    print("Guard:", result["guard_reason"])
    print("Response:", result["response"])
    print(result_metrics_text(result))
    display(Audio(combined_audio, rate=sr))
    return result

## 10. Presentation demo

The Gradio interface accepts live microphone input, displays the ASR transcript and assistant response, and returns Amharic speech.

For presentation simplicity, the audio output is returned as a complete response. The displayed first-audio-ready value remains the compute-side latency measured inside the pipeline; it does not include browser playback overhead.

In [ ]:
import gradio as gr


def presentation_demo(audio_path):
    if not audio_path:
        return "", "", "", "Record an utterance first.", None

    result = run_final_s2s(audio_path)
    sr, response_audio = combine_response_audio(result)

    route = result["response_source"]
    if result["guard_reason"]:
        route = f"{route} ({result['guard_reason']})"

    return (
        result["transcript"],
        result["response"],
        route,
        result_metrics_text(result),
        (sr, response_audio),
    )


with gr.Blocks(title="Amharic Speech-to-Speech Assistant") as demo:
    gr.Markdown("# Amharic Speech-to-Speech Assistant")
    gr.Markdown(
        "Speak in Amharic. The system transcribes the recording, applies safety checks, "
        "generates a short response, and synthesizes Amharic speech."
    )

    microphone = gr.Audio(
        sources=["microphone"],
        type="filepath",
        label="Microphone input",
    )
    run_button = gr.Button("Run assistant", variant="primary")

    transcript_box = gr.Textbox(label="ASR transcript", interactive=False)
    response_box = gr.Textbox(label="Assistant response", interactive=False)
    route_box = gr.Textbox(label="Response route", interactive=False)
    metrics_box = gr.Textbox(label="Timing", interactive=False, lines=4)
    response_audio = gr.Audio(label="Assistant audio", autoplay=True)

    run_button.click(
        fn=presentation_demo,
        inputs=microphone,
        outputs=[
            transcript_box,
            response_box,
            route_box,
            metrics_box,
            response_audio,
        ],
    )

demo.launch(share=True)

## Limitations and evaluation notes

- The final V4 adapter improves Amharic language purity and robustness to several ASR errors, but it still produces occasional semantic and multilingual failures.
- The critical-information guard detects explicit corruption markers; it does not detect every plausible-but-wrong ASR transcription.
- P08 exposed a question-mark handling bug in the guard. The bug was corrected and P08 was retained as a failure-discovery case rather than reused as an unseen post-fix sample.
- P09 was the first post-fix final sample. ASR and routing were correct, but the V4 response contained Japanese leakage, so the response was recorded as a language-purity failure.
- The selected six-word time-sliced strategy is designed to reduce compute-side first-audio latency on a single GPU while avoiding partial Amharic word boundaries.
- Final reporting should use repeated warmed runs and fresh recordings, and should distinguish compute-side first-audio-ready latency from actual playback TTFA.